In [ ]:
import csv
import hashlib
import json
import math
import platform
import random
import subprocess
import sys
import tempfile
import time
from dataclasses import asdict
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
from PIL import __version__ as pillow_version
from torch import nn

# Paramètres de provenance
REPO_URL = "https://github.com/MRdrissi/industrial-defect-ai.git"
BRANCH = "epic/03-convolutional-autoencoder"
DATASET_VERSION = "2"

if DATASET_VERSION == "A_RENSEIGNER":
    raise ValueError("Renseigner la version exacte du dataset Kaggle.")

# Clonage isolé, pour éviter les anciens fichiers de la session.
repo = Path(
    tempfile.mkdtemp(prefix="industrial-defect-ai-", dir="/tmp")
) / "repository"

subprocess.run(
    [
        "git", "clone", "--depth", "1",
        "--branch", BRANCH, REPO_URL, str(repo)
    ],
    check=True,
)

sys.path.insert(0, str(repo))

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8-sig"))

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=repo,
    text=True,
).strip()

# Le notebook doit déjà appartenir au commit exécuté.
notebook_path = repo / "notebooks/06-cae-bottle-training.ipynb"

assert notebook_path.is_file(), (
    "Commiter le notebook sur GitHub avant l'entraînement officiel."
)

assert not subprocess.check_output(
    ["git", "status", "--porcelain"],
    cwd=repo,
    text=True,
).strip()

# Configuration Bottle
config_path = repo / "experiments/configs/cae_bottle.json"
config = read_json(config_path)

EXPERIMENT_ID = config["experiment_id"]
training = config["training"]
early_stopping = training["early_stopping"]

MAX_EPOCHS = training["max_epochs"]
BATCH_SIZE = training["batch_size"]
LEARNING_RATE = training["learning_rate"]
SEED = training["seed"]
PATIENCE = early_stopping["patience"]
MIN_DELTA = early_stopping["min_delta"]

assert EXPERIMENT_ID == "EXP-001"
assert config["category"] == "bottle"
assert training["optimizer"] == "Adam"
assert training["loss"] == "MSELoss"

assert early_stopping["monitor"] == "val_mse"
assert early_stopping["mode"] == "min"
assert early_stopping["restore_best_weights"] is True

assert MAX_EPOCHS > 0
assert PATIENCE > 0
assert MIN_DELTA >= 0

# Dataset
dataset_root = Path("/kaggle/input/datasets/ipythonx/mvtec-ad")
assert (dataset_root / "bottle/train/good").is_dir()

# Vérifier le manifeste officiel de l'EPIC-02.
manifest_path = repo / config["manifest"]
split_protocol = read_json(
    repo / "docs/data/us-02-03/split_protocol.json"
)

expected_hash = split_protocol["manifests"]["bottle"]["sha256"]
manifest_hash = sha256(manifest_path)

assert manifest_hash == expected_hash, (
    "Le manifeste Bottle ne correspond pas au split officiel."
)

# Vérification GPU
assert torch.cuda.is_available(), (
    "Active le GPU dans Kaggle Settings > Accelerator."
)

device = torch.device("cuda")

print("Experiment:", EXPERIMENT_ID)
print("Git commit:", COMMIT)
print("Dataset version:", DATASET_VERSION)
print("Max epochs:", MAX_EPOCHS)
print("Patience:", PATIENCE)
print("Min delta:", MIN_DELTA)
print("GPU:", torch.cuda.get_device_name(0))
print("Configuration et provenance : OK")

In [ ]:
from src.data.mvtec_dataset import MVTecDataset
from src.data.transforms import MVTecTransform
from src.data.dataloaders import create_dataloader

from src.models.convolutional_autoencoder import (
    CAEConfig,
    ConvolutionalAutoencoder,
)

# Charger la configuration de l'architecture.
model_config_path = repo / config["model_config"]
model_parameters = read_json(model_config_path)["config"]

model_parameters["image_size"] = tuple(
    model_parameters["image_size"]
)
model_parameters["encoder_channels"] = tuple(
    model_parameters["encoder_channels"]
)

model_config = CAEConfig(**model_parameters)

transform = MVTecTransform(
    size=model_config.image_size,
    profile="cae",
)

train_dataset = MVTecDataset(
    dataset_root=dataset_root,
    manifest_path=manifest_path,
    category="bottle",
    split="train",
    transform=transform,
)

val_dataset = MVTecDataset(
    dataset_root=dataset_root,
    manifest_path=manifest_path,
    category="bottle",
    split="val",
    transform=transform,
)

# Contrôles de sécurité
assert len(train_dataset) == 167
assert len(val_dataset) == 42

assert all(s["label"] == 0 for s in train_dataset.samples)
assert all(s["label"] == 0 for s in val_dataset.samples)

train_paths = {s["image_path"] for s in train_dataset.samples}
val_paths = {s["image_path"] for s in val_dataset.samples}

assert train_paths.isdisjoint(val_paths)

# Vérifier la disponibilité de toutes les images.
for sample in train_dataset.samples + val_dataset.samples:
    assert (dataset_root / sample["image_path"]).is_file()

# Vérifier les transformations sur une image.
sample = train_dataset[0]

assert sample["image"].shape == (3, 256, 256)
assert sample["image"].dtype == torch.float32
assert 0 <= sample["image"].min().item()
assert sample["image"].max().item() <= 1
assert sample["label"] == 0
assert not sample["mask"].any()

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Train/Validation disjoints: OK")
print("Prétraitement CAE: OK")

In [ ]:
# Reproductibilité
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# DataLoaders
train_loader = create_dataloader(
    train_dataset,
    batch_size=BATCH_SIZE,
    num_workers=0,
    seed=SEED,
)

val_loader = create_dataloader(
    val_dataset,
    batch_size=BATCH_SIZE,
    num_workers=0,
    seed=SEED,
)

# Nouveau modèle : aucun poids pré-entraîné
model = ConvolutionalAutoencoder(model_config).to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

# Créer un dossier unique pour chaque exécution.
# Aucun résultat précédent n'est écrasé.
output_dir = Path(
    tempfile.mkdtemp(
        prefix=f"cae_bottle_{EXPERIMENT_ID}_",
        dir="/kaggle/working",
    )
)

print("Dossier de l'expérience :", output_dir)

checkpoint_path = output_dir / "bottle_best.pth"
history_path = output_dir / "history.json"

history = []

best_val_mse = float("inf")
best_epoch = None

# Référence pour les améliorations significatives
significant_best = float("inf")
epochs_without_improvement = 0

stop_reason = "max_epochs"

print("Device:", device)
print("Paramètres entraînables:", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Initialisation : OK")

In [ ]:
for epoch in range(1, MAX_EPOCHS + 1):

    start_time = time.perf_counter()

    # =====================================
    # TRAIN
    # =====================================

    model.train()
    train_loss_sum = 0.0

    for batch in train_loader:
        images = batch["image"].to(device)

        optimizer.zero_grad(set_to_none=True)

        reconstructed = model(images)
        loss = criterion(reconstructed, images)

        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * images.size(0)

    train_mse = train_loss_sum / len(train_dataset)

    # =====================================
    # VALIDATION
    # =====================================

    model.eval()
    val_loss_sum = 0.0

    with torch.inference_mode():
        for batch in val_loader:
            images = batch["image"].to(device)

            reconstructed = model(images)
            loss = criterion(reconstructed, images)

            val_loss_sum += loss.item() * images.size(0)

    val_mse = val_loss_sum / len(val_dataset)

    if not math.isfinite(train_mse) or not math.isfinite(val_mse):
        raise RuntimeError("Loss non finie détectée.")

    duration = time.perf_counter() - start_time

    # =====================================
    # SAUVEGARDER LE MEILLEUR CHECKPOINT
    # =====================================

    is_best = val_mse < best_val_mse

    if is_best:
        best_val_mse = val_mse
        best_epoch = epoch

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "model_config": asdict(model_config),
                "epoch": epoch,
                "train_mse": train_mse,
                "val_mse": val_mse,
                "git_commit": COMMIT,
                "manifest_sha256": manifest_hash,
                "dataset_version": DATASET_VERSION,
            },
            checkpoint_path,
        )

    # =====================================
    # EARLY STOPPING
    # =====================================

    if val_mse < significant_best - MIN_DELTA:
        significant_best = val_mse
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    # =====================================
    # HISTORIQUE
    # =====================================

    history.append({
        "epoch": epoch,
        "train_mse": train_mse,
        "val_mse": val_mse,
        "duration_seconds": duration,
        "epochs_without_improvement": epochs_without_improvement,
    })

    # Enregistrer les métriques après chaque epoch.
    history_path.write_text(
        json.dumps(history, indent=2),
        encoding="utf-8",
    )

    marker = " [BEST]" if is_best else ""

    print(
        f"Epoch {epoch:03d}/{MAX_EPOCHS} | "
        f"Train: {train_mse:.6f} | "
        f"Val: {val_mse:.6f} | "
        f"Patience: {epochs_without_improvement}/{PATIENCE} | "
        f"{duration:.1f}s{marker}",
        flush=True,
    )

    if epochs_without_improvement >= PATIENCE:
        stop_reason = "early_stopping"

        print(
            f"\nEarly Stopping à l'epoch {epoch}. "
            f"Meilleure epoch : {best_epoch}"
        )
        break

print("\nEntraînement terminé")
print("Epochs exécutées:", len(history))
print("Motif d'arrêt:", stop_reason)
print("Meilleure epoch:", best_epoch)
print(f"Meilleure Validation MSE: {best_val_mse:.6f}")

In [ ]:
assert checkpoint_path.is_file()
assert best_epoch is not None

# Recharger depuis le fichier enregistré.
checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True,
)

assert checkpoint["git_commit"] == COMMIT
assert checkpoint["manifest_sha256"] == manifest_hash

restored_model = ConvolutionalAutoencoder(
    model_config
).to(device)

restored_model.load_state_dict(
    checkpoint["model_state_dict"]
)

restored_model.eval()

# Vérifier la MSE de Validation sans réentraînement.
restored_loss_sum = 0.0

with torch.inference_mode():
    for batch in val_loader:
        images = batch["image"].to(device)

        reconstructed = restored_model(images)
        loss = criterion(reconstructed, images)

        restored_loss_sum += loss.item() * images.size(0)

restored_val_mse = restored_loss_sum / len(val_dataset)

assert math.isclose(
    restored_val_mse,
    best_val_mse,
    rel_tol=1e-4,
    abs_tol=1e-6,
), "La restauration ne reproduit pas la Validation MSE."

print("Checkpoint:", checkpoint_path)
print("Meilleure epoch:", checkpoint["epoch"])
print(f"MSE originale: {best_val_mse:.6f}")
print(f"MSE restaurée: {restored_val_mse:.6f}")
print("Restauration sans réentraînement : OK")

In [ ]:
epochs = [row["epoch"] for row in history]
train_losses = [row["train_mse"] for row in history]
val_losses = [row["val_mse"] for row in history]

plt.figure(figsize=(9, 5))

plt.plot(
    epochs,
    train_losses,
    label="Train MSE",
)

plt.plot(
    epochs,
    val_losses,
    label="Validation MSE",
)

plt.axvline(
    best_epoch,
    linestyle="--",
    label=f"Best epoch: {best_epoch}",
)

plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("CAE Bottle - Train / Validation MSE")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

curve_path = output_dir / "loss_curves.png"
plt.savefig(curve_path, dpi=150)
plt.show()
plt.close()

# Historique CSV
csv_path = output_dir / "history.csv"

with csv_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=history[0].keys(),
    )

    writer.writeheader()
    writer.writerows(history)

print("Courbe:", curve_path)
print("Historique JSON:", history_path)
print("Historique CSV:", csv_path)

In [ ]:
# Charger le template défini par l'EPIC-01.
template_path = (
    repo / "experiments/templates/experiment_template.json"
)

metadata = read_json(template_path)

# =====================================
# IDENTIFICATION
# =====================================

metadata["experiment_id"] = EXPERIMENT_ID
metadata["model"] = "ConvolutionalAutoencoder V1"
metadata["dataset"] = "MVTec AD"
metadata["category"] = "bottle"

# =====================================
# ENVIRONNEMENT
# =====================================

metadata["environment"].update({
    "platform": "Kaggle",
    "python_version": platform.python_version(),
    "torch_version": str(torch.__version__),
    "torchvision_version": torchvision.__version__,
    "numpy_version": np.__version__,
    "pillow_version": pillow_version,
    "matplotlib_version": matplotlib.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "cuda_version": torch.version.cuda,
    "cudnn_version": torch.backends.cudnn.version(),
})

# =====================================
# ENTRAÎNEMENT
# =====================================

metadata["training"].update({
    "epochs": len(history),
    "max_epochs": MAX_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "image_size": list(model_config.image_size),
    "seed": SEED,
    "optimizer": "Adam",
    "loss": "MSELoss",
    "train_images": len(train_dataset),
    "val_images": len(val_dataset),
    "architecture": asdict(model_config),
    "early_stopping": early_stopping,
    "stop_reason": stop_reason,
})

# =====================================
# REPRODUCTIBILITÉ
# =====================================

metadata["reproducibility"].update({
    "git_commit": COMMIT,
    "git_branch": BRANCH,
    "dataset_source": config["dataset_source"],
    "dataset_version": DATASET_VERSION,
    "split_manifest": config["manifest"],
    "split_manifest_sha256": manifest_hash,
    "model_config": config["model_config"],
    "model_config_sha256": sha256(model_config_path),
    "experiment_config": "experiments/configs/cae_bottle.json",
    "experiment_config_sha256": sha256(config_path),
    "notebook": "notebooks/06-cae-bottle-training.ipynb",
    "notebook_sha256": sha256(notebook_path),
})

# =====================================
# MÉTRIQUES
# =====================================

# Les métriques de détection du template restent null :
# image_auroc, precision, recall, f1_score, pixel_auroc.

metadata["metrics"].update({
    "best_epoch": best_epoch,
    "best_val_mse": best_val_mse,
    "restored_val_mse": restored_val_mse,
    "train_mse_at_best_epoch": history[best_epoch - 1]["train_mse"],
    "final_train_mse": history[-1]["train_mse"],
    "final_val_mse": history[-1]["val_mse"],
    "total_training_seconds": sum(
        row["duration_seconds"] for row in history
    ),
})

# =====================================
# ARTEFACTS
# =====================================

working_root = Path("/kaggle/working")

metadata["artifacts"].update({
    "model_path": str(checkpoint_path.relative_to(working_root)),
    "model_sha256": sha256(checkpoint_path),
    "history_path": str(history_path.relative_to(working_root)),
    "history_csv_path": str(csv_path.relative_to(working_root)),
    "loss_curve_path": str(curve_path.relative_to(working_root)),
    "kaggle_version_url": None,
})

metadata["notes"] = (
    "CAE Bottle trained on 167 normal images and validated "
    "on 42 disjoint normal images. "
    "Early Stopping monitors Validation MSE. "
    "The best checkpoint was selected without using the official "
    "test split. Kaggle version URL must be completed after "
    "Save & Run All."
)

# Enregistrer les dépendances effectivement utilisées.
freeze_path = output_dir / "pip-freeze.txt"

with freeze_path.open("w", encoding="utf-8") as file:
    subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        stdout=file,
        check=True,
    )

# Écrire les métadonnées officielles.
metadata_path = output_dir / f"{EXPERIMENT_ID}.json"

metadata_path.write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# =====================================
# VÉRIFICATION FINALE
# =====================================

required_files = [
    checkpoint_path,
    history_path,
    csv_path,
    curve_path,
    metadata_path,
    freeze_path,
]

assert all(path.is_file() for path in required_files)

assert metadata["reproducibility"]["git_commit"] == COMMIT
assert metadata["metrics"]["best_epoch"] == best_epoch

print("EXPÉRIENCE TERMINÉE")
print("Experiment:", EXPERIMENT_ID)
print("Git commit:", COMMIT)
print("Epochs exécutées:", len(history))
print("Motif d'arrêt:", stop_reason)
print("Meilleure epoch:", best_epoch)
print(f"Meilleure Validation MSE: {best_val_mse:.6f}")

print("\nFichiers générés :")

for path in required_files:
    print(" -", path.relative_to(working_root))

print("\nMétadonnées conformes au template : OK")
print("Artefacts présents dans la session : OK")